# Pathology Extraction Monitor

**Keep this notebook running in JupyterLab to prevent Vertex AI idle timeout!**

Runs Phase 1 (rad_pathology_txt → `pathology_extracted`) and Phase 2 (synoptic_report → `pathology_synoptic_extracted`) LLM extractions via Vertex AI, with a live monitor that doubles as a keep-alive.

Same pattern as `dap_training_monitor.ipynb`.

In [ ]:
import os
import sys
import json
import time
import subprocess
from datetime import datetime
from pathlib import Path
from IPython.display import clear_output

In [ ]:
# Configuration — hardcoded to the installed location so it works whether the
# notebook is opened via the symlink at /home/jupyter/pathology_extraction_monitor.ipynb
# (kernel CWD = /home/jupyter/) or from inside the pathology_extraction/ folder.
WORK_DIR = Path('/home/jupyter/pathology_extraction')
assert WORK_DIR.exists(), f'{WORK_DIR} missing — check install'

PROJECT  = os.environ.get('VERTEX_PROJECT') or subprocess.check_output(
    ['gcloud', 'config', 'get-value', 'project']).decode().strip()
LOCATION = os.environ.get('VERTEX_LOCATION', 'us-central1')
os.environ['VERTEX_PROJECT']  = PROJECT
os.environ['VERTEX_LOCATION'] = LOCATION

WORKERS          = 32
REFRESH_INTERVAL = 60   # seconds between dashboard updates (= keep-alive heartbeat)

PHASE1_INPUT  = WORK_DIR / 'phase1_remaining_inputs.jsonl'
PHASE1_OUTPUT = WORK_DIR / 'phase1_outputs.jsonl'
PHASE1_LOG    = WORK_DIR / 'phase1.log'

PHASE2_INPUT  = WORK_DIR / 'synoptic_inputs.jsonl'
PHASE2_OUTPUT = WORK_DIR / 'synoptic_outputs.jsonl'
PHASE2_LOG    = WORK_DIR / 'phase2.log'

print(f'WORK_DIR: {WORK_DIR}')
print(f'Project:  {PROJECT}')
print(f'Location: {LOCATION}')
print(f'Workers:  {WORKERS}')
print()
print(f'Phase 1 input: {PHASE1_INPUT.exists()} ({PHASE1_INPUT.stat().st_size/1e6:.1f} MB)' if PHASE1_INPUT.exists() else f'Phase 1 input MISSING at {PHASE1_INPUT}')
print(f'Phase 2 input: {PHASE2_INPUT.exists()} ({PHASE2_INPUT.stat().st_size/1e6:.1f} MB)' if PHASE2_INPUT.exists() else f'Phase 2 input MISSING at {PHASE2_INPUT}')


## Install deps + verify Vertex auth

In [ ]:
!pip install --quiet google-genai pydantic

In [ ]:
from google import genai
client = genai.Client(vertexai=True, project=PROJECT, location=LOCATION)
r = client.models.generate_content(model='gemini-2.5-flash',
                                    contents='Reply with the single word OK.')
print('Vertex reply:', (r.text or '').strip())

## Launch extractions as background subprocesses

Both scripts are resume-safe — they skip records already present in the output JSONL.
Re-running this cell is harmless; it won't spawn a duplicate if the process is alive.

In [ ]:
def running_pid(cmd_substr):
    try:
        out = subprocess.check_output(['pgrep', '-f', cmd_substr], stderr=subprocess.DEVNULL).decode()
        for line in out.strip().split('\n'):
            if line:
                return int(line)
    except subprocess.CalledProcessError:
        pass
    return None

def launch(script, input_path, output_path, log_path, tag):
    if running_pid(script):
        print(f'[{tag}] already running (pid {running_pid(script)}) — skipping launch')
        return
    if not input_path.exists():
        print(f'[{tag}] input {input_path.name} not found — skip')
        return
    cmd = [
        sys.executable, str(WORK_DIR / script),
        '--input',  str(input_path),
        '--output', str(output_path),
        '--workers', str(WORKERS),
        '--vertex', '--project', PROJECT, '--location', LOCATION,
    ]
    proc = subprocess.Popen(cmd, stdout=open(log_path, 'a'), stderr=subprocess.STDOUT)
    print(f'[{tag}] launched pid {proc.pid} → log {log_path.name}')

launch('run_phase1_jsonl.py', PHASE1_INPUT, PHASE1_OUTPUT, PHASE1_LOG, 'Phase 1')
launch('run_synoptic.py',     PHASE2_INPUT, PHASE2_OUTPUT, PHASE2_LOG, 'Phase 2')

## Run the Monitor

**Execute the cell below to start monitoring.** This will keep refreshing and prevent Vertex AI from timing out.

When both phases hit 100% the loop exits automatically. Press the stop button to exit early.

In [ ]:
def line_count(path):
    if not path.exists(): return 0
    try:
        return int(subprocess.check_output(['wc', '-l', str(path)]).decode().split()[0])
    except Exception:
        return 0

def error_count(path):
    if not path.exists(): return 0
    n = 0
    with open(path) as f:
        for line in f:
            if '"error"' in line:
                try:
                    o = json.loads(line)
                    if o.get('error'): n += 1
                except Exception:
                    pass
    return n

def tail(path, n=8):
    if not path.exists(): return []
    try:
        out = subprocess.check_output(['tail', '-n', str(n), str(path)]).decode()
        return [l for l in out.splitlines() if l.strip()]
    except Exception:
        return []

def phase_block(tag, script, input_path, output_path, log_path, prev_n, dt_sec):
    target = line_count(input_path)
    n = line_count(output_path)
    errs = error_count(output_path)
    rate = (n - prev_n) / max(dt_sec, 1e-6)
    pct = 100 * n / max(target, 1)
    remain = max(target - n, 0)
    eta_min = remain / max(rate, 1e-6) / 60 if rate > 0 else float('inf')
    pid = running_pid(script)
    alive = pid is not None
    bar_len = 30
    filled = int(bar_len * (n / max(target, 1))) if target else 0
    bar = '[' + ('=' * filled).ljust(bar_len) + ']'

    print(f'  {tag:<9}  {bar}  {n:>6,}/{target:<6,} ({pct:>5.1f}%)')
    print(f'             rate={rate:>5.2f}/s  errors={errs:<4}  '
          f'ETA={("%.0fm" % eta_min) if eta_min < 999 else "?"}  '
          f'proc={"alive pid="+str(pid) if alive else "DEAD"}')
    lines = tail(log_path, 3)
    if lines:
        for line in lines[-2:]:
            print(f'             log: {line[:110]}')
    print()
    return n

print('Starting pathology extraction monitor...')
print('Press Ctrl+C or stop the kernel to exit.\n')

prev = {'Phase 1': 0, 'Phase 2': 0}
last_ts = time.time()

while True:
    try:
        clear_output(wait=True)
        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f"\n{'=' * 70}")
        print(f'  Pathology Extraction Monitor — {now}')
        print(f"{'=' * 70}\n")

        dt = max(time.time() - last_ts, 1e-6)
        prev['Phase 1'] = phase_block('Phase 1', 'run_phase1_jsonl.py',
                                       PHASE1_INPUT, PHASE1_OUTPUT, PHASE1_LOG,
                                       prev['Phase 1'], dt)
        prev['Phase 2'] = phase_block('Phase 2', 'run_synoptic.py',
                                       PHASE2_INPUT, PHASE2_OUTPUT, PHASE2_LOG,
                                       prev['Phase 2'], dt)

        both_done = (line_count(PHASE1_OUTPUT) >= line_count(PHASE1_INPUT)
                     and line_count(PHASE2_OUTPUT) >= line_count(PHASE2_INPUT))

        print(f"{'=' * 70}")
        if both_done:
            print('  ✓ Both extractions complete.')
            print(f"{'=' * 70}\n")
            break
        print(f'  Next refresh in {REFRESH_INTERVAL} seconds...')
        print(f'  (Keep this cell running to prevent idle timeout)')
        print(f"{'=' * 70}\n")

        last_ts = time.time()
        time.sleep(REFRESH_INTERVAL)
    except KeyboardInterrupt:
        print('\nMonitor stopped. Extraction subprocesses continue running.')
        break

## Manual Commands

Run these cells manually for one-time checks:

In [ ]:
# Check if extractions are running
!ps aux | grep -E 'run_phase1_jsonl|run_synoptic' | grep -v grep

In [ ]:
# Tail Phase 1 log
!tail -30 {PHASE1_LOG}

In [ ]:
# Tail Phase 2 log
!tail -30 {PHASE2_LOG}

In [ ]:
# Count completed + errored in the JSONL outputs
!echo 'Phase 1:' && wc -l {PHASE1_OUTPUT} 2>/dev/null; grep -c '"error"' {PHASE1_OUTPUT} 2>/dev/null
!echo 'Phase 2:' && wc -l {PHASE2_OUTPUT} 2>/dev/null; grep -c '"error"' {PHASE2_OUTPUT} 2>/dev/null

In [ ]:
# Peek at a few extractions
!head -3 {PHASE2_OUTPUT} 2>/dev/null | python -m json.tool --no-ensure-ascii 2>/dev/null | head -80

## Upload results to GCS when complete

Edit `BUCKET` then run.

In [ ]:
BUCKET = 'gs://REPLACE-ME/pathology_extraction'

if 'REPLACE-ME' in BUCKET:
    print('Set BUCKET to your GCS bucket path, then re-run.')
else:
    for p in [PHASE1_OUTPUT, PHASE2_OUTPUT, PHASE1_LOG, PHASE2_LOG]:
        if p.exists():
            !gsutil cp {p} {BUCKET}/{p.name}
            print(f'Uploaded {p.name}')

## Shut down the VM when fully done

n2-standard-8 ≈ $0.40/hr. Uncomment + run.

In [ ]:
# !gcloud compute instances stop aif-vertex-cpu-us-central1-b --zone=us-central1-b